# Dataframe Merging

Reads daily OHLCV klines CSVs for 8 cryptocurrencies (ADA, BCH, TRX, ETH, BNB, SOL, XRP, BTC) from `klines csv data/`, extracts close price and timestamp, sorts chronologically, and saves each coin as a cleaned CSV to `klines csv data/prices_cleaned/`.

**Date range:** April 2022 – December 2025 (~3.75 years, 1,371 daily candles per coin).  

## How the cleaned data is used downstream

| File | Data structure | Purpose |
|------|----------------|---------|
| `Covariance_Features_DCC_GARCH_Daily.ipynb` | Single merged df (1 column per coin) | Fits DCC-GARCH model to estimate dynamic covariance matrices |
| `Covariance_LSTM.ipynb` | Single merged df (1 column per coin) | Trains an LSTM to predict rolling covariance/correlation matrices |
| `Benchmarks_final.ipynb` | Single merged df (1 column per coin) | Computes 1/N and mean-variance benchmark portfolio performance |
| `CMVO.ipynb` | Single merged df (1 column per coin) | Runs the full CMVO portfolio optimisation using XGBoost/ARIMA return forecasts and DCC-GARCH/LSTM covariance forecasts |
| `XGBoost_Final_All_3_Models.ipynb` | Per-coin (loops over files) | Trains XGBoost models (base, features, Bayesian-optimised) to forecast each coin's returns |
| `arima_order_selection.py` | Per-coin (loops over files) | Selects optimal ARIMA orders for each coin |
| `arima_7d_rebalance_test.py` | Per-coin (loops over files) | Generates ARIMA return forecasts for portfolio rebalancing |


In [1]:
import os
import pandas as pd

In [5]:
import os

folder_path = 'klines csv data'

try:
    os.chdir(folder_path)
    print(f"Successfully changed directory to: {os.getcwd()}")
    print("Contents of the directory:")
    for item in os.listdir('.'):
        print(item)
except FileNotFoundError:
    print(f"Error: The folder '{folder_path}' was not found. Please ensure the path is correct and your Google Drive is mounted.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully changed directory to: c:\Users\ameli\Documents\all ur code projects\optimising-dynamic-crypto-portfolio\klines csv data
Contents of the directory:
ADAUSDT
BCHUSDT
BNBUSDT
BTCUSDT
ETHUSDT
prices_cleaned
prices_cleaned_with_volume
SOLUSDT
TRXUSDT
XRPUSDT


In [11]:
for currency in ("ADAUSDT", "BCHUSDT", "TRXUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT", "BTCUSDT"):
  counter = 0
  prices_df = pd.DataFrame(columns=["close", "time"])
  if currency == "Dataframe Merging.ipynb":
    break
  print(f"Reading Currency: {currency}")
  print("++++++++++++++++"*2)
  for file in os.listdir(os.path.join(os.getcwd(), currency, "2022-04-01_2026-04-01")):
    #print("Reading file: " + str(file))
    temp_df = pd.read_csv(os.path.join(os.getcwd(), currency, "2022-04-01_2026-04-01", str(file)), header = None, usecols=[4,6], names=["close", "time"])
    prices_df = pd.concat([prices_df, temp_df])

  prices_df["time"] = ((prices_df["time"])//1000) - 1648857599
  prices_df = prices_df.sort_values(by="time")

  print("")

  print(prices_df)
  print(f"saved to folder {currency}")
  counter += 1
  prices_df.to_csv(f"prices_cleaned/{currency}")
  print(counter)



Reading Currency: ADAUSDT
++++++++++++++++++++++++++++++++

     close           time
0    1.165              0
1    1.155          86400
2    1.186         172800
3    1.212         259200
4    1.171         345600
..     ...            ...
26    0.37  1765231142400
27  0.3688  1765317542400
28  0.3532  1765403942400
29  0.3514  1765490342400
30  0.3335  1765576742400

[1371 rows x 2 columns]
saved to folder ADAUSDT
1
Reading Currency: BCHUSDT
++++++++++++++++++++++++++++++++

    close           time
0   377.2              0
1   372.5          86400
2   378.6         172800
3   376.2         259200
4   365.4         345600
..    ...            ...
26  623.4  1765231142400
27  622.4  1765317542400
28  599.2  1765403942400
29  595.8  1765490342400
30  599.5  1765576742400

[1371 rows x 2 columns]
saved to folder BCHUSDT
1
Reading Currency: TRXUSDT
++++++++++++++++++++++++++++++++

      close           time
0   0.07494              0
1   0.07322          86400
2   0.07312         17280

In [12]:
print(prices_df.head())

      close    time
0  46283.49       0
1   45811.0   86400
2  46407.35  172800
3  46580.51  259200
4  45497.55  345600
